In [25]:

from mmdet.registry import VISUALIZERS
import sys
from pathlib import Path
import torch 

sys.path.append('/Data_large/marine/PythonProjects/MMDET/notebooks/Tools')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs/custom_components')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')


from mmdet.apis import init_detector, inference_detector
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmcv.transforms import Compose

from cam import EigenCAM
import rasterio as rio 
import numpy as np

def estimate_vmin_vmax(cam, percentiles=[5, 95]):
    """
    Estimates vmin and vmax values for the heatmap.

    Parameters:
    - cam (numpy.ndarray): The heatmap.

    Returns:
    - tuple: A tuple containing the vmin and vmax values.
    """
    vmin = np.percentile(cam, percentiles[0])
    vmax = np.percentile(cam, percentiles[1])
    return vmin, vmax

# Required by loader
def read_tif(file_path, band_indices):
    """
    Reads specified bands from a TIFF file.

    Parameters:
    - file_path (str): Path to the .tif file.
    - band_indices (list of int): Indices of the bands to read.

    Returns:
    - numpy.ndarray: A numpy array containing the stacked band data.
    """
    data = []
    with rio.open(file_path) as src:
        for index in band_indices:
            data.append(src.read(index))
    stacked_data = np.stack(data, axis=0)
    return np.transpose(stacked_data, (1, 2, 0))

DATA_PATH_VEN = '/Data_large/marine/Datasets/VENuS/ds_L0/perfect'
DATA_PATH_SEN = '/Data_large/marine/Datasets/VDS2Raw/imgs'

TIFF_VEN = list(Path(DATA_PATH_VEN).rglob('*.tif'))
TIFF_SEN = list(Path(DATA_PATH_SEN).rglob('*.tif'))


# Specify the path to model config and checkpoint file
config_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/42_BS_3_LR_0.0009_ME_30_OPT_SGD/vfnet_r18.py'
checkpoint_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/42_BS_3_LR_0.0009_ME_30_OPT_SGD/epoch_30.pth'

# build the model from a config file and a checkpoint file
DetModel = init_detector(config_file, checkpoint_file, device='cuda:0')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

cfg = Config.fromfile(config_file)
cfg = cfg.copy()
test_pipeline = cfg.test_dataloader.dataset.pipeline

model = DetModel
checkpoint = checkpoint_file
checkpoint = load_checkpoint(model, checkpoint, map_location='cpu')


checkpoint_meta = checkpoint.get('meta', {})
dataset_meta = checkpoint_meta['dataset_meta']['classes']
model.dataset_meta = dataset_meta

model.to(device)
model.eval()

Idx = 157
tiffSel = None
test_pipeline = Compose(test_pipeline)
if tiffSel is None:
    data_ = dict(img_path=TIFF_VEN[Idx], img_id=0)
    ORIGINAL_IMG = read_tif(file_path=TIFF_VEN[Idx], band_indices=[5])

else:
    data_ = dict(img_path=tiffSel, img_id=0)
    ORIGINAL_IMG = read_tif(file_path=tiffSel, band_indices=[5])
    

data_ = test_pipeline(data_)
data_['inputs'] = [data_['inputs']]
data_['data_samples'] = [data_['data_samples']]


# forward the model
with torch.no_grad():
    results = model.test_step(data_)[0]

visualizer = VISUALIZERS.build(model.cfg.visualizer)

visualizer.dataset_meta = model.dataset_meta

visualizer.dataset_meta = dict(
            classes=('Vessel', ), palette=[
                (
                    220,
                    20,
                    60,
                ),
            ])

visualizer.dpi = 100

scale = 8

visualizer.width *= scale
visualizer.height *= scale

visualizer.line_width = 1

visualizer.alpha = 0.9


visualizer.add_datasample(
name='result',
image=ORIGINAL_IMG,
data_sample=results,
draw_gt=False,
pred_score_thr=0.5,
show=False)

Loads checkpoint by local backend from path: /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/42_BS_3_LR_0.0009_ME_30_OPT_SGD/epoch_30.pth
Loads checkpoint by local backend from path: /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/42_BS_3_LR_0.0009_ME_30_OPT_SGD/epoch_30.pth


/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


In [12]:
dir(visualizer)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_dataset_meta',
 '_default_font_size',
 '_draw_instances',
 '_draw_panoptic_seg',
 '_draw_sem_seg',
 '_image',
 '_init_manager',
 '_initialize_fig',
 '_instance_dict',
 '_instance_name',
 '_is_posion_valid',
 '_vis_backends',
 'add_config',
 'add_datasample',
 'add_graph',
 'add_image',
 'add_scalar',
 'add_scalars',
 'alpha',
 'ax_save',
 'bbox_color',
 'check_instance_created',
 'close',
 'dataset_meta',
 'dpi',
 'draw_bboxes',
 'draw_binary_masks',
 'draw_circles',
 'draw_featmap',
 'draw_lines',
 'draw_points',
 'draw_polygons',
 'draw_texts',
 'fig_save',
 'fig_save_canvas',
 'fig_save_cfg',
 'fig_show_cfg',
 'get_b

In [23]:
visualizer.draw_texts?

Signature:
visualizer.draw_texts(
    texts: Union[str, List[str]],
    positions: Union[numpy.ndarray, torch.Tensor],
    font_sizes: Union[int, List[int], NoneType] = None,
    colors: Union[str, tuple, List[str], List[tuple]] = 'g',
    vertical_alignments: Union[str, List[str]] = 'top',
    horizontal_alignments: Union[str, List[str]] = 'left',
    font_families: Union[str, List[str]] = 'sans-serif',
    bboxes: Union[dict, List[dict], NoneType] = None,
    font_properties: Union[ForwardRef('FontProperties'), List[ForwardRef('FontProperties')], NoneType] = None,
) -> 'Visualizer'
Docstring:
Draw single or multiple text boxes.

Args:
    texts (Union[str, List[str]]): Texts to draw.
    positions (Union[np.ndarray, torch.Tensor]): The position to draw
        the texts, which should have the same length with texts and
        each dim contain x and y.
    font_sizes (Union[int, List[int]], optional): The font size of
        texts. ``font_sizes`` can have the same length with texts 

In [26]:
# import matplotlib.pyplot as plt
import cv2

img = visualizer.get_image()
# save img with cv2
cv2.imwrite(f'/Data_large/marine/PythonProjects/MMDET/results/inference/{Idx}.png', img)


True

In [ ]:
x = data_['inputs']
inp = x[0].to(device).unsqueeze(0)
inp.shape

Forward pass and export to onnx

In [ ]:
# Forward pass
with torch.no_grad():
    outputs = model(inp)


# Export the model to ONNX format
torch.onnx.export(model, inp, 'model.onnx', opset_version=11)

In [ ]:
for o in outputs[0]:
    print(o.shape)

In [ ]:
results